In [8]:
import cv2
import numpy as np
from ultralytics import YOLO
from cv2_enumerate_cameras import enumerate_cameras


class nasp:
    
    def __init__(self):
        self.path = r"C:\Users\Heath Dingus\runs\detect\train-48\weights\best.pt"
        self.model = YOLO(self.path)
        
    @staticmethod
    def name():
        return 'NASP Target'
    
    def get_score(self, distance, target_ref_size):
        if target_ref_size <= 0: return "N/A"
        ratio = distance / target_ref_size
        if ratio < .025: 
            return '10 (X)'
        if ratio < .05:  
            return 10
        if ratio < .10:  
            return 9
        if ratio < .15:  
            return 8
        if ratio < .20:  
            return 7
        if ratio < .25:  
            return 6
        if ratio < .30:  
            return 5
        if ratio < .35:  
            return 4
        if ratio < .40:  
            return 3
        if ratio < .45:
            return 2
        if ratio < .50:
            return 1
        else:
            return 'M (0)'

class OutdoorBarebowRecurve:
        
    def __init__(self):
        self.path = r'C:\Users\Heath Dingus\runs\detect\train-49\weights\best.pt'
        self.model = YOLO(self.path)

    @staticmethod
    def name():
        return '122cm Outdoor Target'
    
    def get_score(self, distance, target_ref_size):
        if target_ref_size <= 0: return "N/A"
        ratio = distance / target_ref_size
        if ratio < .025: 
            return '10 (X)'
        if ratio < .05:  
            return 10
        if ratio < .10:  
            return 9
        if ratio < .15:  
            return 8
        if ratio < .20:  
            return 7
        if ratio < .25:  
            return 6
        if ratio < .30:  
            return 5
        if ratio < .35:  
            return 4
        if ratio < .40:  
            return 3
        if ratio < .45:
            return 2
        if ratio < .50:
            return 1
        else:
            return 'M (0)'




targets = {
    '1': OutdoorBarebowRecurve,
    '2': nasp
}

def select_camera():
    s = set()
    cameras = list(enumerate_cameras())
    if not cameras: return 0
    for camera in cameras:
        if camera.name not in s:
            print(f'Index: {camera.index} | Name: {camera.name}') 
            s.add(camera.name)
    return int(input('Select camera index: '))

def select_target():
    for key, name in targets.items():
        print(f"[{key}] {name.name()}")
    choice = input("Select target index: ")
    return targets.get(choice)()

active_target  = select_target()

cam_index = select_camera()
cap = cv2.VideoCapture(cam_index)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)

frame_count = 0
last_results = None

cv2.namedWindow('LaneAssist', cv2.WINDOW_NORMAL)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    frame_count += 1
    
    if frame_count % 2 == 0 or last_results is None:
        results = active_target.model(frame, imgsz = 1024, conf = 0.4, verbose = False, max_det = 20)
        last_results = results[0]
    
    annotated_frame = frame.copy()
    h_p, w_p, _ = frame.shape
    
    current_center = None
    target_w, target_h = 0, 0
    arrow_data = []

    if last_results.boxes:
        for box in last_results.boxes:
            coords = box.xywhn[0].tolist() 
            cls = int(box.cls[0])
            
            if cls == 1: # Center Ring
                current_center = (coords[0], coords[1])
            elif cls == 0: # Arrow
                arrow_data.append((coords[0], coords[1]))
            elif cls == 2: # Target Face
                target_w, target_h = coords[2], coords[3]

    if current_center and target_w > 0 and target_h > 0:
        aspect_correction = target_h / target_w 
        
        for arrow_xy in arrow_data:
            dx = (arrow_xy[0] - current_center[0]) * aspect_correction
            dy = (arrow_xy[1] - current_center[1])
            dist = np.sqrt(dx**2 + dy**2)
            score = active_target.get_score(dist, target_h)
            
            pix_x, pix_y = int(arrow_xy[0] * w_p), int(arrow_xy[1] * h_p)
            cv2.putText(annotated_frame, f'SCORE: {score}', (pix_x, pix_y - 25), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            
    cv2.imshow('LaneAssist', annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

[1] 122cm Outdoor Target
[2] NASP Target


Select target index:  2


Index: 1400 | Name: e2eSoft iVCam
Index: 1401 | Name: DroidCam Video


Select camera index:  1401
